# HDD 21 — Regime stability and SMART-feature audit

Notebook 20 found 2–4 fitted clusters per HDD model, but persistent clusters could reflect fixed drive age or sector status. Toshiba had low silhouette; ST12000NM0008 had very little persistence above the shuffled control. Before combining regime scores with anomaly detection, check **cluster sizes, sensitivity to drive resampling, and association with SMART 9, temperature, and sector condition**. Use only frozen fit-drive inputs and labels from Notebook 20. No checkpoint inference or Q3 archive scan is needed. Failure flags, evaluation drives, Q4 and sealed June remain unopened.

These are descriptive diagnostics. A cluster correlated with age is not automatically invalid; one tied mostly to sector deterioration must not be treated as a clean normal state. Three drive-resample fits estimate stability under a change of fit drives; they do not replace an external validation or prove a regime has causal meaning.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib,json
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN
from sklearn.metrics import adjusted_rand_score,adjusted_mutual_info_score
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
TRAIN_ID='q3_five_model_masked_transformer_v1'
REGIME_ID='q3_frozen_encoder_regime_feasibility_v1'
AUDIT_ID='q3_regime_stability_smart_audit_v1'
TRAIN=COMPONENT_ROOT/'results'/RUN_ID/TRAIN_ID
REGIME=COMPONENT_ROOT/'results'/RUN_ID/REGIME_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
OUT.mkdir(parents=True,exist_ok=True)
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):
    assert path.is_file(),f'Missing prerequisite: {path}'
    return json.loads(path.read_text())
up=read_json(REGIME/'regime_feasibility_complete.json')
regime_settings=read_json(REGIME/'settings.json')
trained=read_json(TRAIN/'encoder_training_complete.json')
assert up['analysis_complete'] and trained['analysis_complete']
assert not up['failure_labels_used'] and not up['evaluation_drives_used']
assert not up['future_test_opened'] and not up['sealed_June_test_opened']
assert not trained['failure_labels_used']
assert sha256(REGIME/'settings.json')==up['settings_sha256']
embed_path=REGIME/'three_snapshot_fit_embeddings.parquet'
assignment_path=REGIME/'fit_regime_assignments.csv'
spec_path=REGIME/'regime_projection_spec.json'
assert sha256(embed_path)==up['embeddings_sha256']
assert sha256(assignment_path)==up['assignments_sha256']
assert sha256(TRAIN/'fit_windows.npz')==trained['fit_windows_sha256']==regime_settings['fit_windows_sha256']
embeddings=pd.read_parquet(embed_path)
assignments=pd.read_csv(assignment_path,dtype={'serial_number':str})
spec=read_json(spec_path)
models=list(spec)
assert len(models)==5 and set(embeddings.model)==set(models)==set(assignments.model)
assert not embeddings.duplicated(['serial_number','end_date']).any()
assert not assignments.duplicated(['serial_number','end_date']).any()
keys=['serial_number','model','end_date','subset']
joined=embeddings.merge(assignments,on=keys,validate='one_to_one',how='outer',indicator=True)
assert len(joined)==len(embeddings)==len(assignments) and joined._merge.eq('both').all()
joined.drop(columns='_merge',inplace=True)
features=[col for col in embeddings if col.startswith('embed_')]
assert len(features)==64
with np.load(TRAIN/'fit_windows.npz') as a:
    raw=a['windows'];ids=a['serial_number'].astype(str);names=a['model'].astype(str)
    assert a['features'].tolist()==['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
assert raw.shape==(len(ids),30,5) and len(set(ids))==len(ids)
assert set(ids)==set(joined.loc[joined.end_date.eq('2021-08-15'),'serial_number'])
SETTINGS={'notebook20_complete_sha256':sha256(REGIME/'regime_feasibility_complete.json'),
          'notebook20_embeddings_sha256':sha256(embed_path),
          'notebook20_assignments_sha256':sha256(assignment_path),
          'notebook20_projection_spec_sha256':sha256(spec_path),
          'fit_windows_sha256':sha256(TRAIN/'fit_windows.npz'),
          'fit_date':'2021-08-15','seeds':[41,42,43],
          'drive_resample_fraction':.80,'retrain_preprocessing':False,
          'age_bins':'fit-train SMART 9 quintiles, fitted independently by model',
          'sector_condition':'any of SMART 5, 197, 198 is positive',
          'failure_labels_used':False,'winner_selection_enabled':False}
settings_path=OUT/'settings.json'
if settings_path.exists():assert read_json(settings_path)==SETTINGS,'Change AUDIT_ID before changing the protocol.'
else:settings_path.write_text(json.dumps(SETTINGS,indent=2))
print('Inputs verified: frozen embeddings, cluster labels, and August 15 raw fit windows.')

Inputs verified: frozen embeddings, cluster labels, and August 15 raw fit windows.


## 1. Join cluster IDs to raw SMART context

Notebook 17’s saved August 15 windows supply the observed last-day values. Summarize cluster size, drive age, temperature, sector prevalence, and sector changes. These are attributes used by the encoder; this audit tests what the cluster IDs mostly encode. Do not call a cluster healthy, failed, or normal from this table.

In [2]:
context=pd.DataFrame({'serial_number':ids,'model':names,
                      'smart_5_raw_last':raw[:,-1,0],
                      'smart_9_raw_last':raw[:,-1,1],
                      'smart_194_raw_last':raw[:,-1,2],
                      'smart_197_raw_last':raw[:,-1,3],
                      'smart_198_raw_last':raw[:,-1,4],
                      'sector_30day_increase':np.maximum(raw[:,-1,[0,3,4]]-raw[:,0,[0,3,4]],0).sum(axis=1)})
context['sector_positive']=context[['smart_5_raw_last','smart_197_raw_last','smart_198_raw_last']].gt(0).any(axis=1)
fit=joined.loc[joined.end_date.eq('2021-08-15')].merge(context,on=['serial_number','model'],validate='one_to_one')
assert len(fit)==len(ids) and set(fit.serial_number)==set(ids)
assert fit.loc[fit.subset.eq('train'),'hdbscan_fit_label'].notna().all()
assert fit.loc[fit.subset.eq('stop'),'hdbscan_fit_label'].isna().all()
fit['cluster']=fit['hdbscan_fit_label'].fillna(fit['projected_regime']).astype(int)
cluster_summary=fit.groupby(['model','subset','cluster'],dropna=False).agg(
    drives=('serial_number','size'),
    median_power_on_hours=('smart_9_raw_last','median'),
    iqr_power_on_hours=('smart_9_raw_last',lambda s:float(s.quantile(.75)-s.quantile(.25))),
    median_temperature=('smart_194_raw_last','median'),
    sector_positive_fraction=('sector_positive','mean'),
    sector_30day_increase_fraction=('sector_30day_increase',lambda s:float((s>0).mean()))
).reset_index()
cluster_summary['share_within_model_subset']=cluster_summary.drives/cluster_summary.groupby(['model','subset']).drives.transform('sum')
cluster_summary.to_csv(OUT/'cluster_smart_composition.csv',index=False)
fit[['serial_number','model','subset','cluster','smart_9_raw_last','smart_194_raw_last',
     'sector_positive','sector_30day_increase']].to_csv(OUT/'fit_drive_regime_context.csv',index=False)
display(cluster_summary.loc[cluster_summary.subset.eq('train')])

,model,subset,cluster,drives,median_power_on_hours,iqr_power_on_hours,median_temperature,sector_positive_fraction,sector_30day_increase_fraction,share_within_model_subset
4,HGST HMS5C4040BLE640,train,-1,184,39200.5,13028.25,27.0,0.315217,0.097826,0.033774
5,HGST HMS5C4040BLE640,train,0,80,62930.0,7.00,24.0,0.000000,0.000000,0.014684
6,HGST HMS5C4040BLE640,train,1,2953,39353.0,1024.00,25.0,0.000000,0.000000,0.542034
7,HGST HMS5C4040BLE640,train,2,2231,46578.0,1183.50,28.0,0.000000,0.000000,0.409508
11,ST12000NM0008,train,-1,23,14532.0,2582.50,31.0,1.000000,0.130435,0.002812
12,ST12000NM0008,train,0,8069,12262.0,4388.00,29.0,0.000868,0.000868,0.986430
13,ST12000NM0008,train,1,88,14506.5,3525.25,30.0,1.000000,0.204545,0.010758
18,ST4000DM000,train,-1,508,50293.5,7371.25,23.0,0.299213,0.076772,0.062485
19,ST4000DM000,train,0,170,51306.5,3775.50,24.0,0.988235,0.223529,0.020910
20,ST4000DM000,train,1,7107,50585.0,3831.00,22.0,0.000000,0.000000,0.874170


## 2. Quantify simple-feature association and out-of-fit coverage

For training drives with a non-noise HDBSCAN label, report adjusted mutual information (AMI) with age quintile, temperature quintile, and a binary sector flag. AMI measures association, not cause. Quantiles are learned from the fit training subset only. Also compare cluster proportions and sector prevalence on the independent stop subset under Notebook 20’s documented centroid/radius projection. A one-cluster or heavily imbalanced assignment has limited information even if stable.

In [3]:
assoc=[]
for model in models:
    f=fit.loc[fit.model.eq(model)&fit.subset.eq('train')].copy()
    s=fit.loc[fit.model.eq(model)&fit.subset.eq('stop')].copy()
    assert len(f)>100 and len(s)>10
    assigned=f.loc[f.cluster.ne(-1)].copy()
    assert len(assigned)>0
    def bins(col):
        edges=np.unique(np.quantile(f[col].to_numpy(dtype=float),[0,.2,.4,.6,.8,1]))
        if len(edges)<3:return None
        return np.searchsorted(edges[1:-1],assigned[col].to_numpy(dtype=float),side='right')
    age=bins('smart_9_raw_last');temp=bins('smart_194_raw_last')
    labels=assigned.cluster.to_numpy(dtype=int)
    age_ami=float(adjusted_mutual_info_score(labels,age)) if age is not None and len(set(labels))>1 else np.nan
    temp_ami=float(adjusted_mutual_info_score(labels,temp)) if temp is not None and len(set(labels))>1 else np.nan
    sector_ami=float(adjusted_mutual_info_score(labels,assigned.sector_positive.astype(int))) if len(set(labels))>1 else np.nan
    proportions=assigned.cluster.value_counts(normalize=True)
    assoc.append({'model':model,'fit_train_drives':len(f),'stop_drives':len(s),
                  'fit_assigned_clusters':len(proportions),'largest_fit_cluster_share_assigned':float(proportions.max()),
                  'fit_noise_fraction':float(f.cluster.eq(-1).mean()),
                  'stop_projected_noise_fraction':float(s.cluster.eq(-1).mean()),
                  'age_quintile_AMI':age_ami,'temperature_quintile_AMI':temp_ami,'sector_flag_AMI':sector_ami,
                  'fit_sector_positive_fraction':float(f.sector_positive.mean()),
                  'stop_sector_positive_fraction':float(s.sector_positive.mean())})
association=pd.DataFrame(assoc)
association.to_csv(OUT/'regime_feature_associations.csv',index=False)
display(association)

,model,fit_train_drives,stop_drives,fit_assigned_clusters,largest_fit_cluster_share_assigned,fit_noise_fraction,stop_projected_noise_fraction,age_quintile_AMI,temperature_quintile_AMI,sector_flag_AMI,fit_sector_positive_fraction,stop_sector_positive_fraction
0,TOSHIBA MG07ACA14TA,13825,1549,4,0.777368,0.096781,0.077469,0.397650,0.004763,0.000000,0.007884,0.005165
1,ST12000NM0008,8180,864,2,0.989212,0.002812,0.053241,0.002459,0.000292,0.918858,0.014425,0.012731
2,ST4000DM000,8130,871,3,0.932432,0.062485,0.115959,0.080654,0.057333,0.527171,0.039360,0.035591
3,ST8000NM0055,6173,691,4,0.900860,0.039203,0.088278,0.037641,0.006213,0.775733,0.097521,0.125904
4,HGST HMS5C4040BLE640,5448,641,3,0.560980,0.033774,0.076443,0.519781,0.057838,0.000000,0.010646,0.006240


## 3. Drive-resample stability with frozen preprocessing

Keep Notebook 20’s robust scaling and PCA frozen, then refit HDBSCAN on 80% of the original August 15 training drives for three fixed seeds. Compare each resampled label set with Notebook 20’s original labels on **the same selected drives** using adjusted Rand index (ARI). Report ARI both including and excluding noise, plus cluster count and noise fraction. This intentionally tests whether clusters survive a change in drives rather than using HDBSCAN’s deterministic seed on unchanged data. These fits never alter the frozen checkpoint or Notebook 20 outputs.

In [4]:
stability=[]
for model in models:
    f=fit.loc[fit.model.eq(model)&fit.subset.eq('train')].sort_values('serial_number').copy()
    original=f.hdbscan_fit_label.to_numpy(dtype=int)
    config=spec[model]
    x=f[features].to_numpy(dtype=float)
    center=np.asarray(config['scaler_center'],dtype=float)
    scale=np.asarray(config['scaler_scale'],dtype=float)
    pca_mean=np.asarray(config['pca_mean'],dtype=float)
    components=np.asarray(config['pca_components'],dtype=float)
    assert x.shape[1]==len(center)==len(scale)==len(pca_mean)
    assert components.shape[1]==len(center)
    transformed=((x-center)/scale-pca_mean)@components.T
    assert np.isfinite(transformed).all()
    for seed in SETTINGS['seeds']:
        rng=np.random.default_rng(seed)
        chosen=np.sort(rng.choice(len(f),size=int(np.floor(len(f)*SETTINGS['drive_resample_fraction'])),replace=False))
        labels=HDBSCAN(min_cluster_size=config['min_cluster_size'],min_samples=config['min_samples'],
                       allow_single_cluster=False).fit_predict(transformed[chosen])
        orig=original[chosen]
        both=(orig!=-1)&(labels!=-1)
        stability.append({'model':model,'seed':seed,'resampled_drives':len(chosen),
                          'original_cluster_count_on_sample':len(set(orig)-{-1}),
                          'resampled_cluster_count':len(set(labels)-{-1}),
                          'resampled_noise_fraction':float((labels==-1).mean()),
                          'ari_including_noise':float(adjusted_rand_score(orig,labels)),
                          'both_assigned_fraction':float(both.mean()),
                          'ari_both_assigned':float(adjusted_rand_score(orig[both],labels[both])) if both.sum()>1 else np.nan})
        print('Refit',model,'seed',seed,'clusters',len(set(labels)-{-1}))
stability=pd.DataFrame(stability)
stability.to_csv(OUT/'drive_resample_cluster_stability.csv',index=False)
medians=stability.groupby('model')[['ari_including_noise','ari_both_assigned','both_assigned_fraction','resampled_noise_fraction']].median().reset_index()
display(medians)

c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


Refit TOSHIBA MG07ACA14TA seed 41 clusters 4


c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


Refit TOSHIBA MG07ACA14TA seed 42 clusters 4


c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


...

c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


Refit HGST HMS5C4040BLE640 seed 42 clusters 3


c:\Users\IMAKA\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


Refit HGST HMS5C4040BLE640 seed 43 clusters 3


,model,ari_including_noise,ari_both_assigned,both_assigned_fraction,resampled_noise_fraction
0,HGST HMS5C4040BLE640,0.994528,1.0,0.965581,0.034190
1,ST12000NM0008,0.237717,0.0,0.950642,0.049358
2,ST4000DM000,0.982958,1.0,0.933272,0.066728
3,ST8000NM0055,0.988901,1.0,0.944309,0.051235
4,TOSHIBA MG07ACA14TA,0.946454,1.0,0.887071,0.110036


## 4. Research decision record

This audit reports the evidence without silently selecting an encoder, a cluster count, or a final alert rule. In particular, the proposal’s ARI ≥ 0.70 aspiration is a useful development check, but passing this small resampling test is not final validation. If a model has weak separation, one dominant cluster, or unstable resamples, retain it as an explicit limitation and test simpler controls before treating its regime ID as a detection signal.

In [5]:
result=association.merge(medians,on='model',validate='one_to_one')
result['provisional_stability_ge_0_70']=result.ari_including_noise.ge(.70)
result['multiple_substantial_fit_clusters']=result.largest_fit_cluster_share_assigned.lt(.90)&result.fit_assigned_clusters.ge(2)
result['regime_score_ready_for_blind_test']=False
result.to_csv(OUT/'regime_audit_decisions.csv',index=False)
display(result[['model','fit_assigned_clusters','largest_fit_cluster_share_assigned','age_quintile_AMI',
                'sector_flag_AMI','ari_including_noise','ari_both_assigned',
                'provisional_stability_ge_0_70','multiple_substantial_fit_clusters']])
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'settings_sha256':sha256(settings_path),
        'composition_sha256':sha256(OUT/'cluster_smart_composition.csv'),
        'association_sha256':sha256(OUT/'regime_feature_associations.csv'),
        'stability_sha256':sha256(OUT/'drive_resample_cluster_stability.csv'),
        'decision_sha256':sha256(OUT/'regime_audit_decisions.csv'),
        'failure_labels_used':False,'evaluation_drives_used':False,'future_test_opened':False,
        'sealed_June_test_opened':False,'regime_model_approved':False,
        'limitations':['three resamples are an initial stability audit, not a confidence interval',
                       'SMART feature associations do not identify causal cluster drivers',
                       'August 15 fit windows only; out-of-sample projection still approximate',
                       'no held-out warning comparison for regime distance or transitions']}
(OUT/'regime_audit_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 21 COMPLETE — stability and SMART confounds audited without labels.')
print('Q4 and sealed June unopened. Results:',OUT)

,model,fit_assigned_clusters,largest_fit_cluster_share_assigned,age_quintile_AMI,sector_flag_AMI,ari_including_noise,ari_both_assigned,provisional_stability_ge_0_70,multiple_substantial_fit_clusters
0,TOSHIBA MG07ACA14TA,4,0.777368,0.397650,0.000000,0.946454,1.0,True,True
1,ST12000NM0008,2,0.989212,0.002459,0.918858,0.237717,0.0,False,False
2,ST4000DM000,3,0.932432,0.080654,0.527171,0.982958,1.0,True,False
3,ST8000NM0055,4,0.900860,0.037641,0.775733,0.988901,1.0,True,False
4,HGST HMS5C4040BLE640,3,0.560980,0.519781,0.000000,0.994528,1.0,True,True


NOTEBOOK 21 COMPLETE — stability and SMART confounds audited without labels.
Q4 and sealed June unopened. Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_regime_stability_smart_audit_v1
